In [8]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [9]:
!java -version
!pip show pyspark | head

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
Name: pyspark
Version: 4.0.4
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: http://www.apache.org/licenses/LICENSE-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: py4j
Required-by: dataproc-spark-connect


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder.appName("OpenWeather_Pipeline").getOrCreate()
print("Spark session ready: ", spark.version)


Spark session ready:  4.0.4


In [11]:
import time
import requests

In [12]:
API_KEY = "Your_API_Key"
BASE_URL = "https://api.openweathermap.org/data/2.5"
HEADERS = {"User-Agent" : "aditya-pyspark-pipeline/1.0"}
CITIES = ["Indore", "Delhi", "Mumbai", "Bengaluru", "Gurugram",
          "Jalandhar", "Hyderabad", "Chennai", "Pune", "Kolkata"]

In [13]:
def safe_get(url, params, max_attempts = 4):
  for attempt in range(1 , max_attempts + 1):
    response = requests.get(url, params= params, headers= HEADERS, timeout = 20)

    if response.status_code == 200:
      return response.json()
    if response.status_code == 429:
      wait_seconds = 30 * attempt
      print(f"429 rate limited, waiting {wait_seconds}s")
      time.sleep(wait_seconds)
      continue
    response.raise_for_status()
  raise RuntimeError(f"API request failed after {max_attempts} attempts")

def to_float(x):
    return float(x) if x is not None else None

In [14]:
raw_weather_rows = []
for city in CITIES:
  data = safe_get(f"{BASE_URL}/weather", {"q": city, "appid": API_KEY, "units":"metric"})
  raw_weather_rows.append({
      "city" : city,
      "temp_c": to_float(data.get("main",{}).get("temp")),
      "feels_like_c" : to_float(data.get("main",{}).get("feels_like")),
      "min_temp_c" : to_float(data.get("main",{}).get("temp_min")),
      "max_temp_c": to_float(data.get("main", {}).get("temp_max")),
      "humidity": data.get("main",{}).get("humidity"),
      "wind_speed_ms": to_float(data.get("wind",{}).get("speed")),
      "condition": data.get("weather",[{}])[0].get("main"),
      "description": data.get("weather",[{}])[0].get("description"),
      "recorded_at_unix": data.get("dt")
  })
  print(f"fetched: {city}")
  time.sleep(2)

print(f"Extract - API done. Rows: {len(raw_weather_rows)}")

fetched: Indore
fetched: Delhi
fetched: Mumbai
fetched: Bengaluru
fetched: Gurugram
fetched: Jalandhar
fetched: Hyderabad
fetched: Chennai
fetched: Pune
fetched: Kolkata
Extract - API done. Rows: 10


In [15]:
for row in raw_weather_rows:
    if row["temp_c"] is None:
        print("Data missing for:", row["city"])
print("Check done")

Check done


In [16]:
print(raw_weather_rows[1])

{'city': 'Delhi', 'temp_c': 28.96, 'feels_like_c': 32.64, 'min_temp_c': 28.96, 'max_temp_c': 29.05, 'humidity': 70, 'wind_speed_ms': 3.6, 'condition': 'Clear', 'description': 'clear sky', 'recorded_at_unix': 1791610283}


In [17]:
from google.colab import files
uploaded = files.upload()

Saving city_reference.csv to city_reference.csv


In [18]:
weather_schema = StructType([
    StructField("city", StringType(), True),
    StructField("temp_c", DoubleType(), True),
    StructField("feels_like_c", DoubleType(), True),
    StructField("min_temp_c", DoubleType(), True),
    StructField("max_temp_c", DoubleType(), True),
    StructField("humidity", IntegerType(), True),
    StructField("wind_speed_ms", DoubleType(), True),
    StructField("condition", StringType(), True),
    StructField("description", StringType(), True),
    StructField("recorded_at_unix", IntegerType(), True)
])
weather_df = spark.createDataFrame(raw_weather_rows, schema = weather_schema)

city_df = spark.read.csv("city_reference.csv", header= True,inferSchema=True)

print("weather_df: ")
weather_df.show()
print("city_df: ")
city_df.show()

weather_df: 
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+----------------+
|     city|temp_c|feels_like_c|min_temp_c|max_temp_c|humidity|wind_speed_ms|condition|     description|recorded_at_unix|
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+----------------+
|   Indore|  32.1|       31.05|      32.1|      32.1|      31|          3.6|    Clear|       clear sky|      1791610058|
|    Delhi| 28.96|       32.64|     28.96|     29.05|      70|          3.6|    Clear|       clear sky|      1791610283|
|   Mumbai| 34.99|       38.14|     33.94|     34.99|      43|         4.12|    Clear|       clear sky|      1791609867|
|Bengaluru| 28.17|       28.64|     27.18|     29.36|      50|          3.6|   Clouds| overcast clouds|      1791610203|
| Gurugram| 29.02|       31.88|     28.94|     29.02|      65|          3.6|    Clear|       clear sky|      1791610504|
|Jalandhar| 30.98| 

In [19]:
weather_df = weather_df.withColumn(
    "recorded_at", F.from_unixtime(F.col("recorded_at_unix")).cast("timestamp")
).drop("recorded_at_unix")
# before = weather_df.count()
# weather_df = weather_df.dropDuplicates(["city"])
# print(f"Step 5 (Clean) duplicates removed: {before - weather_df.count()}")
# weather_df = weather_df.fillna({"wind_speed_ms": 0})
# city_df = city_df.withColumn("city", F.trim(F.col("city")))


In [20]:
weather_df.show()

+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+
|     city|temp_c|feels_like_c|min_temp_c|max_temp_c|humidity|wind_speed_ms|condition|     description|        recorded_at|
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+
|   Indore|  32.1|       31.05|      32.1|      32.1|      31|          3.6|    Clear|       clear sky|2026-10-10 05:27:38|
|    Delhi| 28.96|       32.64|     28.96|     29.05|      70|          3.6|    Clear|       clear sky|2026-10-10 05:31:23|
|   Mumbai| 34.99|       38.14|     33.94|     34.99|      43|         4.12|    Clear|       clear sky|2026-10-10 05:24:27|
|Bengaluru| 28.17|       28.64|     27.18|     29.36|      50|          3.6|   Clouds| overcast clouds|2026-10-10 05:30:03|
| Gurugram| 29.02|       31.88|     28.94|     29.02|      65|          3.6|    Clear|       clear sky|2026-10-10 05:35:04|
|Jalandh

In [21]:
before = weather_df.count()
weather_df = weather_df.dropDuplicates(["city"])
print(f"duplicates removed : {before - weather_df.count()}")
print("after removing duplicates : ")
weather_df.show()

duplicates removed : 0
after removing duplicates : 
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+
|     city|temp_c|feels_like_c|min_temp_c|max_temp_c|humidity|wind_speed_ms|condition|     description|        recorded_at|
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+
|Bengaluru| 28.17|       28.64|     27.18|     29.36|      50|          3.6|   Clouds| overcast clouds|2026-10-10 05:30:03|
|  Chennai| 32.27|       39.27|     31.67|     33.99|      67|         2.68|   Clouds|scattered clouds|2026-10-10 05:28:32|
|    Delhi| 28.96|       32.64|     28.96|     29.05|      70|          3.6|    Clear|       clear sky|2026-10-10 05:31:23|
| Gurugram| 29.02|       31.88|     28.94|     29.02|      65|          3.6|    Clear|       clear sky|2026-10-10 05:35:04|
|Hyderabad| 31.23|       31.53|     30.73|     31.23|      42|         4.63|    

In [22]:
weather_df = weather_df.fillna({"wind_speed_ms" : 0})

In [23]:
city_df = city_df.withColumn("city", F.trim(F.col("city")))

In [24]:
joined_table = weather_df.join(city_df, on= "city", how= "inner")
print(f"join successfull rows : {joined_table.count()}")
joined_table.show()

join successfull rows : 10
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+--------------+--------------------------+--------+-------+
|     city|temp_c|feels_like_c|min_temp_c|max_temp_c|humidity|wind_speed_ms|condition|     description|        recorded_at|         state|population_approx_millions|area_km2| region|
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+--------------+--------------------------+--------+-------+
|Bengaluru| 28.17|       28.64|     27.18|     29.36|      50|          3.6|   Clouds| overcast clouds|2026-10-10 05:30:03|     Karnataka|                      13.6|     741|  South|
|  Chennai| 32.27|       39.27|     31.67|     33.99|      67|         2.68|   Clouds|scattered clouds|2026-10-10 05:28:32|    Tamil Nadu|                      11.5|     426|  South|
|    Delhi| 28.96|       32.64|     28.96|     29.05|     

In [25]:
joined_table = joined_table.withColumn("density_per_sq_km", F.round(F.col("population_approx_millions") * 1000000 / F.col("area_km2"), 1))

In [26]:
joined_table.show()

+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+--------------+--------------------------+--------+-------+-----------------+
|     city|temp_c|feels_like_c|min_temp_c|max_temp_c|humidity|wind_speed_ms|condition|     description|        recorded_at|         state|population_approx_millions|area_km2| region|density_per_sq_km|
+---------+------+------------+----------+----------+--------+-------------+---------+----------------+-------------------+--------------+--------------------------+--------+-------+-----------------+
|Bengaluru| 28.17|       28.64|     27.18|     29.36|      50|          3.6|   Clouds| overcast clouds|2026-10-10 05:30:03|     Karnataka|                      13.6|     741|  South|          18353.6|
|  Chennai| 32.27|       39.27|     31.67|     33.99|      67|         2.68|   Clouds|scattered clouds|2026-10-10 05:28:32|    Tamil Nadu|                      11.5|     426|  South|          2699

In [27]:
from pyspark.sql.functions.builtin import when
joined_table = joined_table.withColumn(
    "comfort_level",
    F.when(F.col("temp_c") < 10, "Cold")
    .when(F.col("temp_c") < 20, "Cool")
    .when(F.col("temp_c") < 30, "Pleasant")
    .when(F.col("temp_c") < 40, "Hot")
    .otherwise("Very Hot")
)

In [28]:
joined_table = joined_table.withColumn(
    "heat_exposure_Score", F.round(F.col("temp_c") * F.col("density_per_sq_km") / 1000, 2)
)
print("transformation done")

joined_table.select("city","temp_c","density_per_sq_km","comfort_level","heat_exposure_score").show()

transformation done
+---------+------+-----------------+-------------+-------------------+
|     city|temp_c|density_per_sq_km|comfort_level|heat_exposure_score|
+---------+------+-----------------+-------------+-------------------+
|   Mumbai| 34.99|          34328.4|          Hot|            1201.15|
| Gurugram| 29.02|           1571.0|     Pleasant|              45.59|
|    Delhi| 28.96|          21563.3|     Pleasant|             624.47|
|Bengaluru| 28.17|          18353.6|     Pleasant|             517.02|
|   Indore|  32.1|           6603.8|          Hot|             211.98|
|  Chennai| 32.27|          26995.3|          Hot|             871.14|
|Jalandhar| 30.98|           9000.0|          Hot|             278.82|
|  Kolkata| 28.97|          73658.5|     Pleasant|            2133.89|
|     Pune| 32.86|          22356.5|          Hot|             734.63|
|Hyderabad| 31.23|          16153.8|          Hot|             504.48|
+---------+------+-----------------+-------------+-------

In [29]:
joined_table.createOrReplaceTempView("city_weather")

print(">> Cities ranked by heat exposure score")
spark.sql("""
    SELECT city, region, temp_c, density_per_sq_km, heat_exposure_score
    FROM city_weather
    ORDER BY heat_exposure_score DESC
""").show()

print(">> Count of cities per comfort level")
spark.sql("""
    SELECT comfort_level, COUNT(*) AS n_cities
    FROM city_weather
    GROUP BY comfort_level
""").show()

print(">> Average temperature per region")
spark.sql("""
    SELECT region, ROUND(AVG(temp_c), 1) AS avg_temp_c, COUNT(*) AS n_cities
    FROM city_weather
    GROUP BY region
    ORDER BY avg_temp_c DESC
""").show()


>> Cities ranked by heat exposure score
+---------+-------+------+-----------------+-------------------+
|     city| region|temp_c|density_per_sq_km|heat_exposure_score|
+---------+-------+------+-----------------+-------------------+
|  Kolkata|   East| 28.97|          73658.5|            2133.89|
|   Mumbai|   West| 34.99|          34328.4|            1201.15|
|  Chennai|  South| 32.27|          26995.3|             871.14|
|     Pune|   West| 32.86|          22356.5|             734.63|
|    Delhi|  North| 28.96|          21563.3|             624.47|
|Bengaluru|  South| 28.17|          18353.6|             517.02|
|Hyderabad|  South| 31.23|          16153.8|             504.48|
|Jalandhar|  North| 30.98|           9000.0|             278.82|
|   Indore|Central|  32.1|           6603.8|             211.98|
| Gurugram|  North| 29.02|           1571.0|              45.59|
+---------+-------+------+-----------------+-------------------+

>> Count of cities per comfort level
+-----------

In [31]:
pdf = joined_table.toPandas()
pdf.to_csv("multisource_pipeline.csv", index=False)
print(len(pdf), "rows saved")

from google.colab import files
files.download("multisource_pipeline.csv")

10 rows saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>